# core

> fetching , github, arxiv, url2md and more

core is the fetch-and-read layer. `fetch()` handles any URL from a simple static page to a JavaScript-rendered SPA; `to_md()` strips HTML to clean markdown. The module also covers pagination across JSON APIs, reading arXiv papers and YouTube transcripts, and cloning GitHub repos.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from fastcore.all import Path, L, timed_cache, globtastic, parallel, joins, run, first, AttrDict, ifnone, fdelegates, bind
from fastcore.xdg import xdg_cache_home
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from diskcache import Index
import re, httpx, shutil, time, os
import lxml.etree as etree
from scrapling import Selector
from scrapling.engines.toolbelt.custom import Response
from scrapling.fetchers import Fetcher, StealthyFetcher, DynamicFetcher
from urllib.parse import urlparse, urlencode
import yt_dlp
from litesearch import *

In [ ]:
#| export
@retry(wait=wait_exponential(1,30, min=2),stop=stop_after_attempt(4),
       retry=retry_if_exception_type((httpx.TimeoutException, IOError)))
def http_get(url, **kw):
    r = httpx.get(url, timeout=30, **kw)
    if r.status_code == 429: raise IOError(f"Rate limited (429): {url}")
    return r

@retry(wait=wait_exponential(1,30, min=2),stop=stop_after_attempt(4),
       retry=retry_if_exception_type((httpx.TimeoutException, IOError)))
def http_post(url, **kw):
    r = httpx.post(url, timeout=30, **kw)
    if r.status_code == 429: raise IOError(f"Rate limited (429): {url}")
    return r

def save_path(path=None):
    "Get cache path for `name` (e.g. 'arxiv' or 'fetch')"
    p = xdg_cache_home()/'.fossick'
    if path: p /= path
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

In [ ]:
#| export
def _arun(coro) -> any:
	'Run an async coroutine from sync code, even if already inside an event loop'
	import asyncio
	try: asyncio.get_running_loop()
	except RuntimeError: return L(asyncio.run(coro))
	import concurrent.futures
	with concurrent.futures.ThreadPoolExecutor() as pool: return L(pool.submit(asyncio.run, coro).result())

## Web Fetching

`fetch()` returns a dict with `url`, `status`, `html`, `data` (parsed JSON when the response is JSON), and `xhr` (captured network calls when `capture_xhr=True`). `to_md()` produces clean markdown, optionally extracting just the element matched by a CSS selector.

In [ ]:
#| export
def _wrap_md(text, tag): return f'\n<{tag}>\n{text}\n</{tag}>\n' if text else ''

def _html(res_or_html):
    'Extract HTML string from a Page dict or return input if already a string'
    if isinstance(res_or_html, str): return res_or_html
    elif isinstance(res_or_html, Response):
	    return res_or_html.html if getattr(res_or_html,'html', None) is not None else res_or_html.html_content
    return None

def html2md(s:str, ignore_links=True, ignore_images=False):
    'Convert `s` from HTML to markdown'
    import html2text
    from readability import Document
    o = html2text.HTML2Text(bodywidth=5000)
    o.ignore_links, o.mark_code, o.ignore_images = ignore_links, True, ignore_images
    return o.handle(Document(s).summary(keep_all_images=True)) if s.strip() else ''

def to_md(res_or_html,  # Page dict (from fetch/crawl) or raw HTML string
          sel:str=None,  # CSS selector to extract before conversion; returns '' if no match
          multi:bool=False,  # Return all selector matches joined
          wrap_tag:str=None,  # Wrap each multi-result in <wrap_tag>...</wrap_tag>; only used when multi=True
          ignore_links:bool=True,
          ignore_images:bool=False,
          ) -> str:
    'Convert a Page dict or HTML string to clean markdown'
    html = _html(res_or_html)
    _md = lambda h: html2md(str(h), ignore_links, ignore_images)
    if sel:
	    els = Selector(html).css(sel)
	    mds = L(els).map(lambda m: _md(m.get()))
	    if multi: return joins('\n',mds.map(lambda m: _wrap_md(m, wrap_tag) if wrap_tag else m))
	    html = str(mds[0]) if mds else ''
    return _md(html)

In [ ]:
#| export
def get_page(url, method='GET', payload=None, heavy=False, stealthy=False, **kw) -> Response:
    if method.upper() == 'POST': return Fetcher.post(url, json=payload, **kw)
    if payload: kw['params'] = payload
    if not (heavy or stealthy): return Fetcher.get(url, **kw)
    f = DynamicFetcher.async_fetch if heavy else StealthyFetcher.async_fetch
    return _arun(f(url, **kw))[0]

In [ ]:
r1=get_page('https://httpbin.org/get', verify=False)
r1.__dict__.keys()

[2026-06-23 10:51:43] INFO: Fetched (200) <GET https://httpbin.org/get> (referer: https://www.google.com/)


dict_keys(['status', 'reason', 'cookies', 'headers', 'request_headers', 'history', 'meta', 'request', 'captured_xhr'])

In [ ]:
#| export
def fetch(url:str,                # URL to fetch
          sel:str=None,           # CSS selector to extract (None = full page)
          method:str='GET',       # HTTP method; 'POST' sends payload as JSON body
          payload:dict=None,      # POST body (JSON) or GET query params
          heavy:bool=False,       # Full JS rendering via headless browser
          stealthy:bool=False,    # Anti-bot stealth fetcher (Cloudflare etc.)
          capture_xhr:bool=False, # Intercept XHR/fetch calls; forces heavy=True
          cache:bool=False,       # Cache successful responses to disk by URL+sel
          force:bool=False,       # If True, forces re-fetch even if cached result exists
          **kw,                   # Extra kwargs passed to scrapling (e.g. verify, headers)
         ) -> Response:
    'Fetch `url`, return Page dict {url, status, html, data, xhr} where html is raw response body'
    _key = f"{url}\x00{sel or ''}"
    if force and _key in _fetch_cache: del _fetch_cache[_key]
    if cache and _key in _fetch_cache: return _fetch_cache[_key]
    heavy = heavy or capture_xhr
    if capture_xhr: kw.setdefault('capture_xhr', '.*')
    page = get_page(url, method=method, payload=payload, heavy=heavy, stealthy=stealthy, **kw)
    html = page.html_content
    if sel: html = str(el.get()) if (el:=page.css(sel).first) else ''
    xhr = []
    if capture_xhr:
        for x in page.captured_xhr:
            ct = x.headers.get('content-Type', '')
            try: d = x.json()
            except: d = x.text or x.body.decode(errors='replace')
            xhr.append(AttrDict(url=x.url,content_type=ct, data=d))
    setattr(page, 'xhr', xhr)
    setattr(page, 'html', html)
    if cache and page.status == 200: _fetch_cache[_key] = page
    return page

In [ ]:
#| export
def crawl(start_url:str,               # URL to start from
          sel:str=None,                # CSS selector to extract per page
          follow_sel:str='a[href]',    # CSS selector for links to follow
          same_domain:bool=True,       # Only follow links on same domain
          max_pages:int=10,            # Max pages to visit
          delay:float=0,               # Seconds to wait between requests (polite crawling)
          heavy:bool=False,
          stealthy:bool=False,
          **kw,                        # Extra kwargs passed to scrapling (e.g. verify, timeout)
         ) -> list:
    'Crawl from `start_url`, following `follow_sel` links, return list of Page dicts'
    bd, v, q, res = urlparse(start_url).netloc, set(), [start_url], []
    while q and len(v) < max_pages:
        url = q.pop(0)
        if url in v: continue
        v.add(url)
        if delay and res: time.sleep(delay)
        try:
            pg: Response = fetch(url, heavy=heavy, stealthy=stealthy, **kw)
            if pg.status != 200: continue
            if sel: pg.html = str(el.get()) if (el:= pg.css(sel).first) else ''
            res.append(pg)
            for link in pg.css(follow_sel):
                href = link.attrib.get('href', '')
                if not href or href.startswith(('#', 'javascript:', 'mailto:')): continue
                abs_url = pg.urljoin(href)
                if same_domain and urlparse(abs_url).netloc != bd: continue
                if abs_url not in v: q.append(abs_url)
        except Exception as ex:
	        print(f"Error fetching {url}: {ex}. Continuing crawl.")
	        continue
    return res

In [ ]:
#| export
def get_options(page_or_html,  # Page dict (from fetch) or raw HTML string
                sel:str         # CSS selector for the <select> element
               ) -> list:
    "Extract options from a <select> element; returns [{'value': ..., 'text': ...}]"
    return [{'value': el.attrib.get('value', el.text or ''), 'text': (el.text or '').strip()}
            for el in Selector(_html(page_or_html)).css(f'{sel} option')]

def fetch_all(urls:list,           # URLs to fetch
              sel:str=None,         # CSS selector to extract per page (None = full page)
              concurrency:int=8,    # Max parallel fetches
              heavy:bool=False,
              stealthy:bool=False,
              **kw                  # Extra kwargs passed to fetch()
             ) -> list:
    "Fetch a list of URLs in parallel; returns Page dicts in the same order as urls"
    return parallel(fetch, urls, sel=sel, heavy=heavy, stealthy=stealthy, threadpool=True, n_workers=concurrency, **kw)

In [ ]:
_sel_html = '''<html><body>
<select id="kanda">
  <option value="1">Balakanda</option>
  <option value="2">Ayodhyakanda</option>
  <option value="3">Aranyakanda</option>
</select>
</body></html>'''

opts = get_options(_sel_html, '#kanda')
assert opts == [{'value': '1', 'text': 'Balakanda'},
                {'value': '2', 'text': 'Ayodhyakanda'},
                {'value': '3', 'text': 'Aranyakanda'}]

### arXiv

In [ ]:
#| export
def get_pdf(url:str):
    'Fetch PDF from URL and return as PdfDocument'
    r = get_page(url)
    if r.status != 200: raise Exception(f"Failed to fetch PDF: {r.status}")
    return PdfDocument.from_bytes(r.body)

In [ ]:
#| export
_arxin = Index(str(save_path('arxiv_cache.db')))
_fetch_cache = {}
def _fetch_arxiv_meta(arxiv_id:str):
    "Fetch and parse arxiv metadata for a given ID; result is cached to disk"
    r = http_get(f'https://export.arxiv.org/api/query?id_list={arxiv_id}')
    if r.status_code != 200: raise Exception(f"Failed to fetch arxiv data: {r.status_code}")
    ns = {'a': 'http://www.w3.org/2005/Atom'}
    entry = etree.fromstring(r.content).find('a:entry', ns)
    if entry is None: raise Exception("No paper found")
    def txt(tag): return entry.find(f'a:{tag}', ns).text
    pu = first(L(entry.findall('a:link',ns)).filter(lambda l: l.get('title')=='pdf').map(lambda l: l.get('href')))
    au = L(entry.findall('a:author', ns)).map(lambda a: a.find('a:name', ns).text)
    return dict(title=txt('title').strip(), summary=txt('summary').strip(),
                published=txt('published'), link=txt('id'), pdf_url=pu, authors=au)

In [ ]:
#| export
def read_arxiv(url:str, # arxiv PDF URL, or arxiv abstract URL, or arxiv ID
               save_pdf:bool=True, # if True, saves the downloaded PDF to disk
               save_dir:str='.', # directory in which to save the PDF
               force:bool=False # if True, forces re-download of PDF even if it exists on disk
               ):
    "Get paper information from arxiv URL or ID, optionally saving PDF to disk"
    arxiv_id = url.rsplit('/', 1)[-1]
    ver = re.search(r'v(\d+)$', arxiv_id)
    ver_sfx = f'v{ver.group(1)}' if ver else ''
    arxiv_id = re.sub(r'v\d+$', '', arxiv_id)
    pdf_path = Path(save_dir)/f'{arxiv_id}{ver_sfx}.pdf'
    if not force and arxiv_id in _arxin:
        res = dict(_arxin[arxiv_id])
        if save_pdf and pdf_path.exists(): res['pdf_path'] = pdf_path
        return res
    res = dict(_fetch_arxiv_meta(arxiv_id))
    if not res['pdf_url']: raise Exception("No PDF URL found in arxiv metadata")
    doc = get_pdf(res['pdf_url']) if not pdf_path.exists() else PdfDocument(str(pdf_path))
    if save_pdf:
        if not pdf_path.exists():
            pdf_path.parent.mkdir(parents=True, exist_ok=True)
            doc.save(str(pdf_path), compress=True, garbage_collect=True)
        res['pdf_path'] = pdf_path
    res |= dict(source=doc.to_markdown_all())
    _arxin[arxiv_id] = {k: v for k, v in res.items() if k != 'pdf_path'}
    return res

In [ ]:
read_arxiv('https://arxiv.org/abs/2306.14881')['summary'][:200]

'Low-metallicity dwarf galaxies often show no or little CO emission, despite the intense star formation observed in local samples. Both simulations and resolved observations indicate that molecular gas'

In [ ]:
#| export
def _crossref_search(**params):
    "Search Crossref API; returns parsed response dict"
    return fetch(f'https://api.crossref.org/works?{urlencode(params)}', verify=False).json() or {}

_crossref = bind(_crossref_search, **{'select': 'DOI,title', 'rows': 1})

def lookup_doi(title:str  # Paper title to search
               ) -> str|None:
    "Return doi.org URL for first Crossref match on `title`"
    items = _crossref(**{'query.title': title}).get('message', {}).get('items', [])
    doi = items[0].get('DOI') if items else None
    return f"https://doi.org/{doi}" if doi else None

In [ ]:
lookup_doi('Self-Determination Theory and the Facilitation of Intrinsic Motivation, Social Development, and Well-Being')

[2026-06-23 10:00:09] INFO: Fetched (200) <GET https://api.crossref.org/works?select=DOI%2Ctitle&rows=1&query.title=Self-Determination+Theory+and+the+Facilitation+of+Intrinsic+Motivation%2C+Social+Development%2C+and+Well-Being> (referer: https://www.google.com/)


'https://doi.org/10.1037//0003-066x.55.1.68'

In [ ]:
res = fetch('https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf', verify=False)

[2026-06-23 10:00:13] INFO: Fetched (200) <GET https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf> (referer: https://www.google.com/)


In [ ]:
#| export
from fastcore.nbio import Notebook, mk_cell, new_nb, os

In [ ]:
#| export
def _nb_stem(url):
	"Derive a filesystem-safe notebook stem from a URL"
	path = urlparse(url).path.rsplit('/', 1)[-1].split('.')[0]
	stem = path or urlparse(url).netloc.split('.')[0]
	return re.sub(r'[^\w-]', '_', stem)[:50]

def _reflow(t):
	paras = [p.strip() for p in re.split(r'\n\n+', t) if p.strip()]
	out, buf = [], []
	for p in paras:
		if re.match(r'^#{1,6} |^[-*+] |^\d+\.', p):
			if buf: out.append(' '.join(buf)); buf = []
			out.append(p)
		else:
			buf.append(p)
			if re.search(r'[.!?]\s*$', p): out.append(' '.join(buf)); buf = []
	if buf: out.append(' '.join(buf))
	return '\n\n'.join(out)

def _text_segs(t): return [c.strip() for c in re.split(r'\n(?=#{1,3} )', t) if c.strip()]

def _md2cells(md):
	'Split markdown into cells; [code]...[/code] blocks from html2text become code cells'
	code_re = re.compile(r'\[code\]\n\n(.*?)\n\[/code\]', re.DOTALL)
	segments, last_end = [], 0
	for m in code_re.finditer(md):
		text = md[last_end:m.start()].strip()
		if text: segments += [('markdown', s) for s in _text_segs(text)]
		code = re.sub(r'^    ', '', m.group(1), flags=re.MULTILINE).strip()
		if code: segments += [('code', code)]
		last_end = m.end()
	if text := md[last_end:].strip(): segments += [('markdown', s) for s in _text_segs(text)]
	cells = []
	for i, (kind, content) in enumerate(segments):
		if kind == 'code': cells.append(mk_cell(content, 'code'))
		else: cells.append(mk_cell(content, 'markdown'))
	return cells

@fdelegates(fetch)
def url2nb(url, nb_path=None, **kwargs):
	'Convert any URL (PDF, arxiv, or HTML page) to a Jupyter notebook'
	if url.endswith('.pdf'): return pdf2nb(url, nb_path=nb_path, **kwargs)
	if 'arxiv.org' in url:
		res = read_arxiv(url)
		return pdf2nb(str(res['pdf_path']), nb_path=nb_path)
	md = to_md(fetch(url, **kwargs))
	nb_path = Path(nb_path or f'{_nb_stem(url)}.ipynb')
	Notebook(new_nb(cells=_md2cells(md)), path=nb_path).save()
	return nb_path

@fdelegates(fetch)
def pdf2nb(url_or_path, nb_path=None, image_dir='images', **kwargs):
	'Convert PDF to a Jupyter notebook; one markdown cell per page + empty code cell'
	url_or_path = str(url_or_path)
	if not url_or_path.endswith('.pdf'): return None
	if url_or_path.startswith('http'):
		pdf = get_pdf(url_or_path)
		stem = url_or_path.rsplit('/', 1)[-1].replace('.pdf', '')
	else: pdf, stem = PdfDocument(url_or_path), Path(url_or_path).stem
	nb_path = Path(nb_path or f'{stem}.ipynb').resolve()
	cwd = os.getcwd()
	try:
		os.chdir(nb_path.parent)
		Path(image_dir).mkdir(exist_ok=True)
		md = pdf.to_markdown_all(preserve_layout=True, include_images=True,
          embed_images=False, image_output_dir=image_dir)
	finally: os.chdir(cwd)
	Notebook(new_nb(cells=_md2cells(md)), path=nb_path).save()
	return nb_path

In [ ]:
#| eval: False
url2nb('https://squiddev.medium.com/continuing-continuations-cps-in-python-47bba90c8d1e', verify=False, cache=True)

Path('continuing-continuations-cps-in-python-47bba90c8d1.ipynb')

In [ ]:
#| eval: False
url2nb('https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html', verify=False, cache=True)

[2026-06-23 10:00:50] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html> (referer: https://www.google.com/)


Path('thinking_in_jax.ipynb')

In [ ]:
#| eval: False
pdf2nb('https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf', 'sdt-test.ipynb', verify=False, cache=True)

Path('/Users/71293/code/personal/orgs/fossick/nbs/sdt-test.ipynb')

### GitHub

In [ ]:
#| export
def _gh_ssh(url:str): # GitHub URL, SSH remote, or bare `user/repo`
    'Convert GitHub URL to SSH remote; pass through if already SSH; return None if not GitHub'
    if url.startswith('git@github.com:'): return url
    if m := re.match(r'https://github\.com/([^/]+)/([^/]+)', url):
        return f'git@github.com:{m.group(1)}/{m.group(2)}.git'

def _get_git_repo(gh_ssh:str):
    'Clone or update a GitHub repo to local cache; return Path'
    repo_name = gh_ssh.rsplit('/', 1)[-1].removesuffix('.git')
    repo_dir = save_path('git_clones')/repo_name
    if repo_dir.exists():
        try:
            run(['git', '-C', str(repo_dir), 'fetch'])
            branch = run(['git', '-C', str(repo_dir), 'symbolic-ref', 'refs/remotes/origin/HEAD']).split('/')[-1]
            run(['git', '-C', str(repo_dir), 'reset', '--hard', f'origin/{branch}'])
            return repo_dir
        except IOError: shutil.rmtree(repo_dir)
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', gh_ssh, str(repo_dir)])
    return repo_dir

@timed_cache(seconds=3600)
def read_gh_repo(path_or_url:str,  # GitHub URL, SSH address, or local path
                 globs:tuple=None,  # file glob patterns (default: README*, pyproject.toml, *.py)
                 limit:int=None,    # max files to return
                 as_list:bool=False # return list of Paths instead of {path: content} dict
                ):
    'Read files from a GitHub repo filtered by glob patterns'
    ssh = _gh_ssh(path_or_url)
    repo_dir = _get_git_repo(ssh) if ssh else Path(path_or_url)
    if globs is None: globs = ('README*', 'pyproject.toml', '*.py')
    files = L(p for g in globs for p in globtastic(repo_dir, file_glob=g, func=Path)).unique()
    if limit: files = files[:limit]
    if as_list: return files
    return {str(p): p.read_text(errors='ignore') for p in files}

def read_gh_file(url:str # GitHub blob URL of the file to read
                 ):
	'Read raw contents of a file from its GitHub URL'
	raw_url = re.sub(r'https://github\.com/([^/]+)/([^/]+)/blob/([^/]+)/(.+)',
	                 r'https://raw.githubusercontent.com/\1/\2/refs/heads/\3/\4', url)
	if (r:=http_get(raw_url)).status_code != 200: raise Exception(f"Failed to fetch {raw_url}: {r.status_code}")
	return r.text

In [ ]:
read_gh_file('https://github.com/Karthik777/litesearch/blob/main/README.md')[:200]

'# litesearch\n\n\n<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->\n\n> **NB** Reading this on GitHub? The formatted\n> [documentation](https://Karthik777.github.io/litesearch/) is nicer.\n\nlitese'

In [ ]:
#| eval: false
list(read_gh_repo('https://github.com/vedicreader/gheasy'))

['/Users/71293/.cache/.fossick/git_clones/gheasy/README.md',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/pyproject.toml',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/__init__.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/_modidx.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/core.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/workflow.py']

## API Discovery & Pagination

`find_xhr()` visits a page with a real browser, captures all XHR and fetch calls it makes, and returns those matching a URL pattern. This surfaces the undocumented JSON endpoints that JavaScript-heavy sites use to load their data. `paginate_api()` replays one of those captured requests across pages until results are exhausted.

In [ ]:
#| export
def compile_pattern(pattern):
	"Compile pattern as regex; if invalid (e.g. bare glob like *foo*), convert via fnmatch first"
	try: return re.compile(pattern)
	except re.error:
		import fnmatch
		return re.compile(fnmatch.translate(pattern))

def find_xhr(url:str,             # URL to visit with browser
             pattern:str='*',     # Glob or regex pattern to filter captured XHR URLs
             **kw,                # Extra kwargs passed to fetch (verify, network_idle, etc.)
            ) -> list:
    "Visit `url` with a headless browser, return [{url, content_type, data}] for each XHR/fetch call made"
    kw.setdefault('network_idle', True)
    rx = compile_pattern(pattern)
    xhr = fetch(url, capture_xhr=True, **kw).xhr
    filtered = [x for x in xhr if rx.search(x.url)]
    return filtered

In [ ]:
assert compile_pattern('.*products.*').search('https://api.example.com/products?q=1')
assert compile_pattern('*products*').search('https://api.example.com/products?q=1')
assert not compile_pattern('*products*').search('https://api.example.com/search')
assert compile_pattern('.*[Ss]earch.*').search('https://api.example.com/Search')

In [ ]:
#| export
def paginate_api(url:str,                      # API endpoint URL
                 payload:dict=None,            # Request body (POST) or params (GET)
                 page_field:str='pageNumber',  # Payload key to increment for each page
                 size_field:str='pageSize',    # Payload key for page size (detects last page)
                 results_field:str=None,       # Response key with items list (auto-detect if None)
                 method:str='POST',            # HTTP method
                 max_pages:int=10,
                 page_size=24,                 # Page size to request (only used if not in payload)
                 page_start:int=1,             # Starting page number (default 1)
                 save:bool=False,              # If True, saves each page's items to disk
                 save_file:str='{url}_page_{page}.json', # Filepath pattern for saving (only used if save=True)
                 force:bool=False,             # If True, forces re-fetching even if saved file exists
                 **kw,                         # Extra kwargs passed to fetch() (verify, headers, etc.)
                ) -> list:
    "Paginate through a JSON API, collecting all results. Auto-detects the items list in response."
    payload = dict(payload or {})
    page_size = payload.get(size_field, page_size)
    all_items = []
    for page_num in range(page_start, max_pages + page_start):
        sf = save_file.format(url=urlparse(url).netloc, page=page_num)
        if not force and Path(sf).exists():
            print(f"Page {page_num} already saved, skipping fetch")
            with open(sf) as f: all_items.extend(json.load(f)); continue
        if (pg:= fetch(url, method=method, payload={**payload, page_field: page_num}, **kw)).status != 200: break
        try: data = pg.json()
        except: break
        if isinstance(data, list): items = data
        elif results_field:
            raw = data.get(results_field, [])
            items = list(raw.values()) if isinstance(raw, dict) else raw
        else:
            lists = {k: v for k, v in data.items() if isinstance(v, list)}
            items = max(lists.values(), key=len) if lists else []
        if not items: break
        all_items.extend(items)
        if len(items) < page_size: break
        if save: Path(sf).write_text(json.dumps(items, indent=2))
    return all_items

In [ ]:
#| export
def download_files(url:str,
                   pattern:str='*',          # Glob or regex to match download hrefs
                   save_dir:str='.',         # Directory to save files
                   follow:bool=False,        # Crawl linked pages looking for more files
                   follow_sel:str='a[href]', # Selector for page links to follow when crawling
                   file_sel:str='a[href]',   # Selector to scan for download links on each page
                   same_domain:bool=True,
                   max_pages:int=10,
                   delay:float=0,
                   force:bool=False,         # Re-download even if file already exists
                   heavy:bool=False,
                   stealthy:bool=False,
                   **kw,                     # Extra kwargs passed to fetch/http_get (e.g. verify)
                  ) -> L:
    "Download all hrefs matching pattern from url; optionally crawl linked pages first"
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    rx = compile_pattern(pattern)
    pages = (crawl(url, follow_sel=follow_sel, same_domain=same_domain, max_pages=max_pages, delay=delay,
       heavy=heavy, stealthy=stealthy, **kw) if follow else [fetch(url, heavy=heavy, stealthy=stealthy, **kw)])
    saved, seen = L(), set()
    for pg in pages:
        for a in pg.css(file_sel):
            href = a.attrib.get('href', '')
            if not rx.search(href): continue
            abs_url = pg.urljoin(href)
            if abs_url in seen: continue
            seen.add(abs_url)
            fname = save_dir / abs_url.rsplit('/', 1)[-1].split('?')[0]
            if fname.exists() and not force: saved.append(fname); continue
            if (r:=get_page(abs_url, verify=kw.get('verify', False))).status == 200: fname.write_bytes(r.body)
            saved.append(fname)
    return saved

In [ ]:
#| eval: False
download_files('https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html', follow=True, follow_sel='a[class="right-next"]', file_sel='a[href$=".ipynb"]', save_dir='jax_notebooks', verify=False, cache=True)

[2026-06-21 13:08:50] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/_sources/notebooks/thinking_in_jax.ipynb> (referer: https://www.google.com/)
[2026-06-21 13:08:51] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/_sources/notebooks/Common_Gotchas_in_JAX.ipynb> (referer: https://www.google.com/)
[2026-06-21 13:08:52] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/_sources/parallel.ipynb> (referer: https://www.google.com/)


[Path('jax_notebooks/thinking_in_jax.ipynb'), Path('jax_notebooks/thinking_in_jax.ipynb'), Path('jax_notebooks/thinking_in_jax.ipynb'), Path('jax_notebooks/Common_Gotchas_in_JAX.ipynb'), Path('jax_notebooks/Common_Gotchas_in_JAX.ipynb'), Path('jax_notebooks/Common_Gotchas_in_JAX.ipynb'), Path('jax_notebooks/parallel.ipynb'), Path('jax_notebooks/parallel.ipynb')]

In [ ]:
from fastcore.test import test_eq

In [ ]:
_pages = crawl('https://lego.sankalpa.sh/', max_pages=2, verify=False)
assert isinstance(_pages, list), f"Expected list, got {type(_pages)}"
assert len(_pages) > 0, "Expected at least one page"
assert all(p.status == 200 for p in _pages), "Non-200 pages should be skipped"
assert all(len(p.html) > 0 for p in _pages), "html should be non-empty"
assert len({p.url for p in _pages}) == len(_pages), "url values should be unique"

[2026-06-22 20:47:06] INFO: Fetched (200) <GET https://lego.sankalpa.sh/> (referer: https://www.google.com/)


In [ ]:
#| eval: false
# Step 1: visit the listing page with a headless browser, capture all XHR/fetch calls
apis = find_xhr('https://www.danmurphys.com.au/list/wine-all', verify=False)

[2026-06-21 13:19:15] INFO: Fetched (200) <GET https://www.danmurphys.com.au/list/wine-all> (referer: https://www.google.com/)
[2026-06-21 13:19:15] INFO: Fetched (200) <GET https://aem.danmurphys.com.au/list/wine-all.model.json> (referer: https://www.danmurphys.com.au/)
[2026-06-21 13:19:15] ERROR: Error getting page content in async: Response.body: Response body is unavailable for redirect responses
[2026-06-21 13:19:15] INFO: Fetched (302) <GET https://dpm.demdex.net/id?d_visid_ver=5.5.0&d_fieldgroup=MC&d_rtbd=json&d_ver=2&d_verify=1&d_orgid=1124C2D754E497DC0A4C98C6%40AdobeOrg&d_nsid=0&ts=1782011949098> (referer: https://www.danmurphys.com.au/)
[2026-06-21 13:19:15] INFO: Fetched (200) <GET https://dpm.demdex.net/id/rd?d_visid_ver=5.5.0&d_fieldgroup=MC&d_rtbd=json&d_ver=2&d_verify=1&d_orgid=1124C2D754E497DC0A4C98C6%40AdobeOrg&d_nsid=0&ts=1782011949098> (referer: https://www.danmurphys.com.au/)
[2026-06-21 13:19:15] INFO: Fetched (200) <GET https://target.danmurphys.com.au/rest/v1/de

In [ ]:
#| eval: false
# paginate_api: test with JSONPlaceholder (free public REST API, GET-based)
posts = paginate_api(
    'https://jsonplaceholder.typicode.com/posts',
    payload={'_page': 1, '_limit': 5},
    page_field='_page',
    size_field='_limit',
    method='GET',
    verify=False,
)
assert len(posts) >= 5
assert 'title' in posts[0]

[2026-06-03 08:27:32] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=1&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:33] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=2&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:33] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=3&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:34] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=4&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:35] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=5&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:36] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=6&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:36] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=7&_limit=5> (referer: https://www.google.com/)

## YouTube

`search_yt()` runs a YouTube search and returns metadata for each video. `read_yt()` fetches the auto-generated English captions as plain text, disk-cached by video ID. `download_yt()` saves audio or video to disk.

In [ ]:
#| export
_yt_cache = Index(str(save_path('yt_cache.db')))

def _yt_id(url:str) -> str:
    "Extract 11-char video ID from any YouTube URL or pass through bare ID"
    if m := re.search(r'(?:v=|youtu\.be/|/shorts/|/embed/)([A-Za-z0-9_-]{11})(?![A-Za-z0-9_-])', url): return m.group(1)
    return url

def _parse_vtt(vtt:str) -> str:
    "Strip VTT timing lines and cue tags, deduplicate adjacent lines, return plain text"
    lines = []
    for line in vtt.splitlines():
        if (re.match(r'^\d{2}:\d{2}', line) or '-->' in line
                or re.match(r'^(WEBVTT|Kind:|Language:|NOTE)', line)
                or not line.strip()): continue
        text = re.sub(r'<[^>]+>', '', line).strip()
        if text and (not lines or lines[-1] != text): lines.append(text)
    return ' '.join(lines)

In [ ]:
#| export
def search_yt(q:str, n:int=10) -> L:
    "Search YouTube; returns L of dicts: id, title, url, duration, view_count, channel, description, thumbnail"
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'extract_flat': True}) as ydl:
            info = ydl.extract_info(f'ytsearch{n}:{q}', download=False)
        return L({'id': e.get('id'),
                  'title': e.get('title'),
                  'url': e.get('url') or f'https://www.youtube.com/watch?v={e.get("id")}',
                  'duration': e.get('duration'),
                  'view_count': e.get('view_count'),
                  'channel': e.get('channel'),
                  'description': e.get('description'),
                  'thumbnail': next((t['url'] for t in reversed(e.get('thumbnails', [])) if 'url' in t), None)}
                 for e in info.get('entries', []))
    except Exception as e:
        print(f'search_yt error: {e}')
        return L()

In [ ]:
results = search_yt('3blue1brown neural networks', n=3)
assert isinstance(results, L), f"expected L, got {type(results)}"
assert len(results) >= 1, f"expected results, got {len(results)}"
assert any(kw in results[0]['title'].lower() for kw in ('3blue1brown', 'neural', 'network')), \
    f"unexpected title: {results[0]['title']}"
assert results[0]['url'].startswith('https://www.youtube.com'), f"bad url: {results[0]['url']}"
print(results[0]['title'], '|', results[0]['url'])

But what is a neural network? | Deep learning chapter 1 | https://www.youtube.com/watch?v=aircAruvnKk


In [ ]:
#| export
def read_yt(url:str, force:bool=False) -> dict:
    "Fetch YouTube metadata + English transcript (auto-captions); result disk-cached by video ID"
    vid = _yt_id(url)
    if not force and vid in _yt_cache: return _yt_cache[vid]
    with yt_dlp.YoutubeDL({'quiet': True, 'skip_download': True}) as ydl:
        info = ydl.extract_info(f'https://www.youtube.com/watch?v={vid}', download=False)
    caps = info.get('automatic_captions', {})
    en_caps = caps.get('en') or caps.get('en-orig') or []
    vtt_url = next((c['url'] for c in en_caps if c.get('ext') == 'vtt'), None)
    source = ''
    if vtt_url:
        r = http_get(vtt_url)
        if r.status_code == 200: source = _parse_vtt(r.text)
    res = dict(title=info.get('title', ''),
               channel=info.get('channel') or info.get('uploader', ''),
               description=info.get('description', ''),
               duration=info.get('duration'),
               upload_date=info.get('upload_date'),
               source=source)
    _yt_cache[vid] = res
    return res

In [ ]:
meta = read_yt('https://www.youtube.com/watch?v=aircAruvnKk')
assert meta['title'], "title should be non-empty"
assert isinstance(meta['source'], str), "source should be a string"
assert len(meta['source']) > 100, f"transcript too short: {len(meta['source'])} chars"
assert '3blue1brown' in meta['channel'].lower(), f"unexpected channel: {meta['channel']}"
print(f"title: {meta['title']}")
print(f"transcript preview: {meta['source'][:200]}")

title: But what is a neural network? | Deep learning chapter 1
transcript preview: [Music] This is a three. It's sloppily written and rendered at an extremely low resolution of 28x 28 pixels. But your brain has no trouble recognizing it as a three. And I want you to take a moment to


In [ ]:
#| export
def download_yt(url:str, format:str='audio', save_dir:str='.', quality:str=None) -> Path:
    "Download YouTube media; format='audio'|'video'|yt-dlp format string. Returns Path to saved file."
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    outtmpl = str(save_dir / '%(title)s.%(ext)s')
    if format == 'audio':
        opts = {'quiet': True, 'format': 'bestaudio/best', 'outtmpl': outtmpl,
                'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3',
                                    'preferredquality': quality or '192'}]}
    elif format == 'video':
        opts = {'quiet': True, 'format': 'bestvideo+bestaudio/best', 'outtmpl': outtmpl,
                'merge_output_format': 'mp4'}
    else:
        opts = {'quiet': True, 'format': format, 'outtmpl': outtmpl}
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        stem = Path(ydl.prepare_filename(info)).stem
    ext = 'mp3' if format == 'audio' else ('mp4' if format == 'video' else '*')
    candidates = sorted(save_dir.glob(f'{stem}*.{ext}'))
    return candidates[0] if candidates else save_dir / f'{stem}.{ext}'

In [ ]:
#| eval: false
p = download_yt('https://www.youtube.com/watch?v=aircAruvnKk', format='audio', save_dir='/tmp/fossick_test')
assert p.exists(), f"file not found: {p}"
assert p.suffix == '.mp3', f"expected .mp3, got {p.suffix}"
print(f"saved to: {p} ({p.stat().st_size // 1024} KB)")

   saved to: /tmp/fossick_test/But what is a neural network？ ｜ Deep learning chapter 1.mp3 (26250 KB)


## Install

In [ ]:
#| export
def repo_root() -> Path:
    'Find the root of the current git repository, or None if not in a repo.'
    return first((Path.cwd(), *Path.cwd().parents), lambda p: (p/'.git').exists())

def mv_skill_md(dry_run=True, dir=None) -> None:
    'Copy bundled SKILL.md to skill directories.'
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    if not (src := base.joinpath('SKILL.md')).exists(): return
    root = dir or repo_root() or '.'
    ts = [Path(root)/'.agents/skills/fossick/SKILL.md', Path('.claude/skills/fossick/SKILL.md')]
    if dry_run: print(f'Would copy {src} to: {list(map(str,ts))}')
    else:
        [p.mk_write(src.read_text(encoding='utf-8')) for p in ts]
        print(f'Installed → {list(map(str,ts))}')

In [ ]:
root = repo_root()
assert root is not None and (root/'.git').exists(), f"Expected git root, got {root}"
mv_skill_md(dry_run=True)

Would copy /Users/71293/code/personal/orgs/fossick/nbs/SKILL.md to: ['/Users/71293/code/personal/orgs/fossick/.agents/skills/fossick/SKILL.md', '.claude/skills/fossick/SKILL.md']


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()